# 데이터 점검 결과 요약

- `lot_info.csv` 데이터 크기: 150행 × 6열
- `inspection.csv` 데이터 크기: 150행 × 6열
- `defect.csv` 데이터 크기: 136행 × 4열
- `report_template.xlsx` 구성: `품질_보고서` 시트 1개, 주요 KPI 요약·제품별 품질 지표·분석 결과 요약 입력 영역 포함
- 결측값 없음
- 전처리 방향
  - `Lot_ID`를 기준으로 `lot_info.csv`와 `inspection.csv`를 먼저 결합하고, `defect.csv`는 Lot별 불량 집계 결과로 요약한 뒤 결합합니다.
  - `Start_Date`, `End_Date`, `Inspection_Date`는 날짜형으로 변환하고, 검사일이 생산 종료일보다 빠른 Lot이 있는지 점검합니다.
  - `defect.csv`에는 완전 중복 행 1개가 있으므로 중복 제거 여부를 확인한 뒤 Lot별 불량 건수와 주요 불량 유형을 산출합니다.
  - 템플릿의 핵심 입력 항목은 평균 수율, 불량률, 검사 Lot 수, 주요 불량 유형, 수율 미달 Lot 수, 최저 수율 제품, 제품별 검사 Lot 수·평균 수율·불량률입니다.
  - 반도체 품질 분석의 핵심 KPI는 수율과 불량률이므로, 템플릿 작성에 필요한 범위로 Lot·제품 단위 품질 지표를 우선 산출합니다.


### 1단계. 데이터 불러오기와 초기 점검

- 지시: 첨부된 `lot_info.csv`를 `lot_df`, `inspection.csv`를 `inspection_df`, `defect.csv`를 `defect_df`, `report_template.xlsx`를 템플릿 파일로 불러온 뒤, 세 데이터프레임 각각에 대해 미리보기, 정보 확인, 열별 결측값 개수 확인을 수행하세요. 각 셀 단위로 파이썬 코드를 작성하세요.
- 이유: 세 데이터의 구조, 자료형, 결측 상태를 먼저 확인해야 템플릿에 넣을 지표 산출 기준을 안정적으로 정할 수 있기 때문입니다.

In [1]:
# Cell 1
# 이 셀은 필요한 라이브러리를 불러옵니다.
import pandas as pd
from openpyxl import load_workbook

In [2]:
# Cell 2
# 이 셀은 csv 파일 3개와 엑셀 템플릿 파일을 불러옵니다.
lot_df = pd.read_csv(filepath_or_buffer="lot_info.csv")
inspection_df = pd.read_csv(filepath_or_buffer="inspection.csv")
defect_df = pd.read_csv(filepath_or_buffer="defect.csv")
template_wb = load_workbook(filename="report_template.xlsx")

In [3]:
# Cell 3
# 이 셀은 lot_df의 앞부분을 미리 봅니다.
lot_df.head()

,Lot_ID,Product,Fab,Start_Date,End_Date,Input_Wafer_Qty
0,LOT_202603_0001,DRAM_B,FAB2,2026-03-13,2026-03-16,25
1,LOT_202603_0002,Logic_A,FAB1,2026-03-20,2026-03-25,25
2,LOT_202603_0003,NAND_A,FAB1,2026-03-15,2026-03-21,25
3,LOT_202603_0004,DRAM_B,FAB1,2026-03-03,2026-03-09,25
4,LOT_202603_0005,DRAM_A,FAB2,2026-03-23,2026-03-27,25


In [4]:
# Cell 4
# 이 셀은 lot_df의 구조와 자료형을 확인합니다.
lot_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Lot_ID           150 non-null    object
 1   Product          150 non-null    object
 2   Fab              150 non-null    object
 3   Start_Date       150 non-null    object
 4   End_Date         150 non-null    object
 5   Input_Wafer_Qty  150 non-null    int64 
dtypes: int64(1), object(5)
memory usage: 7.2+ KB


In [5]:
# Cell 5
# 이 셀은 lot_df의 열별 결측값 개수를 확인합니다.
lot_df.isna().sum()

Lot_ID             0
Product            0
Fab                0
Start_Date         0
End_Date           0
Input_Wafer_Qty    0
dtype: int64

In [6]:
# Cell 6
# 이 셀은 inspection_df의 앞부분을 미리 봅니다.
inspection_df.head()

,Lot_ID,Inspection_Date,Good_Wafer_Qty,Yield_Rate,Defect_Density,Final_Judgment
0,LOT_202603_0001,2026-03-16,24,96.58,0.171,Pass
1,LOT_202603_0002,2026-03-25,22,91.07,0.300,Hold
2,LOT_202603_0003,2026-03-21,21,87.53,0.277,Hold
3,LOT_202603_0004,2026-03-09,24,97.08,0.214,Pass
4,LOT_202603_0005,2026-03-27,23,94.65,0.194,Pass


In [7]:
# Cell 7
# 이 셀은 inspection_df의 구조와 자료형을 확인합니다.
inspection_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Lot_ID           150 non-null    object 
 1   Inspection_Date  150 non-null    object 
 2   Good_Wafer_Qty   150 non-null    int64  
 3   Yield_Rate       150 non-null    float64
 4   Defect_Density   150 non-null    float64
 5   Final_Judgment   150 non-null    object 
dtypes: float64(2), int64(1), object(3)
memory usage: 7.2+ KB


In [8]:
# Cell 8
# 이 셀은 inspection_df의 열별 결측값 개수를 확인합니다.
inspection_df.isna().sum()

Lot_ID             0
Inspection_Date    0
Good_Wafer_Qty     0
Yield_Rate         0
Defect_Density     0
Final_Judgment     0
dtype: int64

In [9]:
# Cell 9
# 이 셀은 defect_df의 앞부분을 미리 봅니다.
defect_df.head()

,Lot_ID,Defect_Type,Defect_Count,Severity
0,LOT_202603_0002,Open defect,41,Medium
1,LOT_202603_0002,Overlay fail,29,Low
2,LOT_202603_0002,Overlay fail,29,Low
3,LOT_202603_0002,Thickness fail,44,High
4,LOT_202603_0003,CD outlier,7,High


In [10]:
# Cell 10
# 이 셀은 defect_df의 구조와 자료형을 확인합니다.
defect_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 136 entries, 0 to 135
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Lot_ID        136 non-null    object
 1   Defect_Type   136 non-null    object
 2   Defect_Count  136 non-null    int64 
 3   Severity      136 non-null    object
dtypes: int64(1), object(3)
memory usage: 4.4+ KB


In [11]:
# Cell 11
# 이 셀은 defect_df의 열별 결측값 개수를 확인합니다.
defect_df.isna().sum()

Lot_ID          0
Defect_Type     0
Defect_Count    0
Severity        0
dtype: int64

### 2단계. 보고서 템플릿 입력 항목 확인

- 지시: `report_template.xlsx`의 `품질_보고서` 시트를 확인하여 값을 채워야 하는 셀과 항목명을 정리하세요.
- 이유: 템플릿 작성을 최우선 목표로 삼기 위해 필요한 KPI, 표, 요약 문장을 먼저 확정해야 하기 때문입니다.

In [12]:
# Cell 12
# 이 셀은 품질_보고서 시트를 선택합니다.
report_ws = template_wb["품질_보고서"]

In [13]:
# Cell 13
# 이 셀은 특정 셀 기준으로 왼쪽과 위쪽에서 가장 가까운 항목명을 찾는 함수를 만듭니다.
def find_left_label(ws, row, col):
    for c in range(col - 1, 0, -1):
        value = ws.cell(row=row, column=c).value
        if value is not None:
            return value
    return None


def find_upper_label(ws, row, col):
    for r in range(row - 1, 0, -1):
        value = ws.cell(row=r, column=col).value
        if value is not None:
            return value
    return None

In [14]:
# Cell 14
# 이 셀은 값이 비어 있고 주변 항목명이 있는 셀을 찾아 정리합니다.
fill_items = []

for row in range(1, report_ws.max_row + 1):
    for col in range(1, report_ws.max_column + 1):
        cell = report_ws.cell(row=row, column=col)

        if cell.value is None:
            left_label = find_left_label(ws=report_ws, row=row, col=col)
            upper_label = find_upper_label(ws=report_ws, row=row, col=col)

            if left_label is not None or upper_label is not None:
                fill_items.append(
                    {
                        "시트명": "품질_보고서",
                        "입력셀": cell.coordinate,
                        "항목명": left_label if left_label is not None else upper_label,
                        "왼쪽항목": left_label,
                        "위쪽항목": upper_label,
                    }
                )

fill_cells_df = pd.DataFrame(data=fill_items)

In [15]:
# Cell 15
# 이 셀은 값을 채워야 하는 셀과 항목명을 확인합니다.
fill_cells_df

,시트명,입력셀,항목명,왼쪽항목,위쪽항목
0,품질_보고서,B1,반도체 품질 분석 보고서,반도체 품질 분석 보고서,None
1,품질_보고서,C1,반도체 품질 분석 보고서,반도체 품질 분석 보고서,None
2,품질_보고서,D1,반도체 품질 분석 보고서,반도체 품질 분석 보고서,None
3,품질_보고서,B2,보고 기간:,보고 기간:,None
4,품질_보고서,D2,작성일자:,작성일자:,None
5,품질_보고서,A3,보고 기간:,None,보고 기간:
6,품질_보고서,C3,작성일자:,None,작성일자:
7,품질_보고서,B4,주요 KPI 요약,주요 KPI 요약,None
8,품질_보고서,C4,주요 KPI 요약,주요 KPI 요약,작성일자:
9,품질_보고서,D4,주요 KPI 요약,주요 KPI 요약,None


### 3단계. 키와 날짜 정합성 점검

- 지시: `lot_df`와 `inspection_df`에서 `Lot_ID`의 중복 여부와 서로 매칭되지 않는 Lot이 있는지 확인하고, `Start_Date`, `End_Date`, `Inspection_Date`를 날짜형으로 변환한 뒤 검사일이 종료일보다 빠른 Lot이 있는지 점검하세요.
- 이유: Lot 단위 결합과 기간 계산의 오류를 막아야 이후 품질 지표가 잘못 계산되지 않기 때문입니다.

In [16]:
# Cell 16
# 이 셀은 lot_df와 inspection_df에서 Lot_ID 중복 개수를 확인합니다.
lot_duplicate_count = lot_df["Lot_ID"].duplicated().sum()
inspection_duplicate_count = inspection_df["Lot_ID"].duplicated().sum()

pd.Series(
    data={
        "lot_df_Lot_ID_중복개수": lot_duplicate_count,
        "inspection_df_Lot_ID_중복개수": inspection_duplicate_count,
    }
)

lot_df_Lot_ID_중복개수           0
inspection_df_Lot_ID_중복개수    0
dtype: int64

In [17]:
# Cell 17
# 이 셀은 lot_df에는 있지만 inspection_df에는 없는 Lot_ID를 확인합니다.
lot_only = sorted(set(lot_df["Lot_ID"]) - set(inspection_df["Lot_ID"]))

lot_only

[]

In [18]:
# Cell 18
# 이 셀은 inspection_df에는 있지만 lot_df에는 없는 Lot_ID를 확인합니다.
inspection_only = sorted(set(inspection_df["Lot_ID"]) - set(lot_df["Lot_ID"]))

inspection_only

[]

In [19]:
# Cell 19
# 이 셀은 날짜 관련 열을 날짜형으로 변환합니다.
lot_df["Start_Date"] = pd.to_datetime(arg=lot_df["Start_Date"])
lot_df["End_Date"] = pd.to_datetime(arg=lot_df["End_Date"])
inspection_df["Inspection_Date"] = pd.to_datetime(arg=inspection_df["Inspection_Date"])

In [20]:
# Cell 20
# 이 셀은 Lot_ID를 기준으로 종료일과 검사일을 결합합니다.
lot_inspection_df = pd.merge(
    left=lot_df[["Lot_ID", "End_Date"]],
    right=inspection_df[["Lot_ID", "Inspection_Date"]],
    on="Lot_ID",
    how="inner",
)

In [21]:
# Cell 21
# 이 셀은 검사일이 종료일보다 빠른 Lot을 확인합니다.
invalid_inspection_date_df = lot_inspection_df[
    lot_inspection_df["Inspection_Date"].lt(other=lot_inspection_df["End_Date"])
]

invalid_inspection_date_df

,Lot_ID,End_Date,Inspection_Date


### 4단계. 불량 데이터 중복 및 집계 기준 정리

- 지시: `defect_df`에서 완전 중복 행을 확인하고, 중복 제거 전후의 불량 유형별 건수와 총 불량 수 차이를 비교한 뒤 분석에 사용할 `defect_clean_df`를 만드세요.
- 이유: 불량 데이터의 중복 여부가 주요 불량 유형과 불량률 계산에 직접 영향을 주기 때문입니다.

In [22]:
# Cell 22
# 이 셀은 defect_df에서 완전 중복 행을 확인합니다.
duplicated_defect_df = defect_df[defect_df.duplicated(keep=False)]

duplicated_defect_df

,Lot_ID,Defect_Type,Defect_Count,Severity
1,LOT_202603_0002,Overlay fail,29,Low
2,LOT_202603_0002,Overlay fail,29,Low


In [23]:
# Cell 23
# 이 셀은 중복 제거 전의 불량 유형별 건수를 확인합니다.
defect_count_before = defect_df["Defect_Type"].value_counts().sort_index()

defect_count_before

Defect_Type
CD outlier        18
Open defect       15
Overlay fail      18
Particle          17
Pattern bridge    21
Scratch           28
Thickness fail    19
Name: count, dtype: int64

In [24]:
# Cell 24
# 이 셀은 완전 중복 행을 제거하여 분석용 데이터프레임을 만듭니다.
defect_clean_df = defect_df.drop_duplicates().copy()

In [25]:
# Cell 25
# 이 셀은 중복 제거 후의 불량 유형별 건수를 확인합니다.
defect_count_after = defect_clean_df["Defect_Type"].value_counts().sort_index()

defect_count_after

Defect_Type
CD outlier        18
Open defect       15
Overlay fail      17
Particle          17
Pattern bridge    21
Scratch           28
Thickness fail    19
Name: count, dtype: int64

In [26]:
# Cell 26
# 이 셀은 중복 제거 전후의 불량 유형별 건수 차이를 비교합니다.
defect_count_compare_df = pd.DataFrame(
    data={
        "중복제거전": defect_count_before,
        "중복제거후": defect_count_after,
    }
).fillna(value=0).astype(dtype=int)

defect_count_compare_df["차이"] = (
    defect_count_compare_df["중복제거전"]
    - defect_count_compare_df["중복제거후"]
)

defect_count_compare_df

,중복제거전,중복제거후,차이
Defect_Type,,,
CD outlier,18,18,0
Open defect,15,15,0
Overlay fail,18,17,1
Particle,17,17,0
Pattern bridge,21,21,0
Scratch,28,28,0
Thickness fail,19,19,0


In [27]:
# Cell 27
# 이 셀은 중복 제거 전후의 총 불량 수 차이를 비교합니다.
pd.Series(
    data={
        "중복제거전_총불량수": len(defect_df),
        "중복제거후_총불량수": len(defect_clean_df),
        "차이": len(defect_df) - len(defect_clean_df),
    }
)

중복제거전_총불량수    136
중복제거후_총불량수    135
차이              1
dtype: int64

### 5단계. Lot 단위 통합 데이터 생성

- 지시: `defect_clean_df`를 `Lot_ID` 기준으로 총 불량 수, 불량 유형 수, 가장 많이 발생한 불량 유형, High 심각도 불량 수로 집계한 뒤, `lot_df`와 `inspection_df`를 결합하고 이 불량 집계 결과까지 결합하여 `quality_df`를 생성하세요.
- 이유: 템플릿의 KPI와 제품별 지표를 한 데이터프레임에서 일관되게 계산하기 위해 Lot 단위 통합 데이터가 필요하기 때문입니다.

In [28]:
# Cell 28
# 이 셀은 Lot_ID별 불량 집계 결과를 만듭니다.
defect_summary_df = (
    defect_clean_df
    .groupby(by="Lot_ID")
    .agg(
        총불량수=("Defect_Type", "size"),
        불량유형수=("Defect_Type", "nunique"),
        High심각도불량수=("Severity", lambda x: x.eq("High").sum()),
    )
    .reset_index()
)

In [29]:
# Cell 29
# 이 셀은 Lot_ID별 가장 많이 발생한 불량 유형을 집계합니다.
top_defect_type_df = (
    defect_clean_df
    .groupby(by="Lot_ID")["Defect_Type"]
    .agg(lambda x: x.value_counts().idxmax())
    .reset_index(name="최다불량유형")
)

In [30]:
# Cell 30
# 이 셀은 불량 집계 결과와 최다 불량 유형을 결합합니다.
defect_summary_df = pd.merge(
    left=defect_summary_df,
    right=top_defect_type_df,
    on="Lot_ID",
    how="left",
)

In [31]:
# Cell 31
# 이 셀은 lot_df와 inspection_df를 Lot_ID 기준으로 결합합니다.
quality_df = pd.merge(
    left=lot_df,
    right=inspection_df,
    on="Lot_ID",
    how="left",
)

In [32]:
# Cell 32
# 이 셀은 Lot_ID 기준 불량 집계 결과까지 결합하여 quality_df를 완성합니다.
quality_df = pd.merge(
    left=quality_df,
    right=defect_summary_df,
    on="Lot_ID",
    how="left",
)

In [33]:
# Cell 33
# 이 셀은 불량이 없는 Lot의 집계값을 0 또는 없음으로 채웁니다.
quality_df[["총불량수", "불량유형수", "High심각도불량수"]] = (
    quality_df[["총불량수", "불량유형수", "High심각도불량수"]]
    .fillna(value=0)
    .astype(dtype=int)
)

quality_df["최다불량유형"] = quality_df["최다불량유형"].fillna(value="없음")

In [34]:
# Cell 34
# 이 셀은 완성된 quality_df의 앞부분을 확인합니다.
quality_df.head()

,Lot_ID,Product,Fab,Start_Date,End_Date,Input_Wafer_Qty,Inspection_Date,Good_Wafer_Qty,Yield_Rate,Defect_Density,Final_Judgment,총불량수,불량유형수,High심각도불량수,최다불량유형
0,LOT_202603_0001,DRAM_B,FAB2,2026-03-13,2026-03-16,25,2026-03-16,24,96.58,0.171,Pass,0,0,0,없음
1,LOT_202603_0002,Logic_A,FAB1,2026-03-20,2026-03-25,25,2026-03-25,22,91.07,0.300,Hold,3,3,1,Open defect
2,LOT_202603_0003,NAND_A,FAB1,2026-03-15,2026-03-21,25,2026-03-21,21,87.53,0.277,Hold,4,4,2,CD outlier
3,LOT_202603_0004,DRAM_B,FAB1,2026-03-03,2026-03-09,25,2026-03-09,24,97.08,0.214,Pass,0,0,0,없음
4,LOT_202603_0005,DRAM_A,FAB2,2026-03-23,2026-03-27,25,2026-03-27,23,94.65,0.194,Pass,0,0,0,없음


### 6단계. 템플릿용 파생 지표 생성

- 지시: `quality_df`에 투입 웨이퍼 대비 양품 수 기준의 수율 검증값, 불량 여부, 수율 미달 여부, 공정 소요 일수, Lot별 불량률을 계산한 파생 열을 추가하세요.
- 이유: 보고서의 평균 수율, 불량률, 수율 미달 Lot 수를 같은 기준으로 계산하기 위해 파생 지표가 필요하기 때문입니다.

In [35]:
# Cell 35
# 이 셀은 투입 웨이퍼 대비 양품 수 기준의 수율 검증값을 계산합니다.
quality_df["수율검증값"] = (
    quality_df["Good_Wafer_Qty"] / quality_df["Input_Wafer_Qty"] * 100
).round(2)

In [36]:
# Cell 36
# 이 셀은 총 불량 수가 1개 이상이면 불량 여부를 True로 표시합니다.
quality_df["불량여부"] = quality_df["총불량수"].gt(other=0)

In [37]:
# Cell 37
# 이 셀은 수율 검증값이 기준 수율보다 낮은지 확인합니다.
quality_df["수율미달여부"] = quality_df["수율검증값"].lt(other=quality_df["Yield_Rate"])

In [38]:
# Cell 38
# 이 셀은 Lot별 공정 소요 일수를 계산합니다.
quality_df["공정소요일수"] = (
    quality_df["End_Date"] - quality_df["Start_Date"]
).dt.days

In [39]:
# Cell 39
# 이 셀은 Lot별 불량률을 계산합니다.
quality_df["불량률"] = (
    quality_df["총불량수"] / quality_df["Input_Wafer_Qty"] * 100
).round(2)

In [40]:
# Cell 40
# 이 셀은 파생 열이 추가된 quality_df의 앞부분을 확인합니다.
quality_df.head()

,Lot_ID,Product,Fab,Start_Date,End_Date,Input_Wafer_Qty,Inspection_Date,Good_Wafer_Qty,Yield_Rate,Defect_Density,Final_Judgment,총불량수,불량유형수,High심각도불량수,최다불량유형,수율검증값,불량여부,수율미달여부,공정소요일수,불량률
0,LOT_202603_0001,DRAM_B,FAB2,2026-03-13,2026-03-16,25,2026-03-16,24,96.58,0.171,Pass,0,0,0,없음,96.0,False,True,3,0.0
1,LOT_202603_0002,Logic_A,FAB1,2026-03-20,2026-03-25,25,2026-03-25,22,91.07,0.300,Hold,3,3,1,Open defect,88.0,True,True,5,12.0
2,LOT_202603_0003,NAND_A,FAB1,2026-03-15,2026-03-21,25,2026-03-21,21,87.53,0.277,Hold,4,4,2,CD outlier,84.0,True,True,6,16.0
3,LOT_202603_0004,DRAM_B,FAB1,2026-03-03,2026-03-09,25,2026-03-09,24,97.08,0.214,Pass,0,0,0,없음,96.0,False,True,6,0.0
4,LOT_202603_0005,DRAM_A,FAB2,2026-03-23,2026-03-27,25,2026-03-27,23,94.65,0.194,Pass,0,0,0,없음,92.0,False,True,4,0.0


### 7단계. 주요 KPI 요약값 산출

- 지시: `quality_df`를 사용하여 보고 기간, 평균 수율, 전체 불량률, 검사 Lot 수, 주요 불량 유형, 수율 미달 Lot 수, 최저 수율 제품을 계산하여 `kpi_summary`로 정리하세요.
- 이유: 보고서 상단의 주요 KPI 요약 영역을 자동으로 채우기 위한 값이 필요하기 때문입니다.

In [41]:
# Cell 41
# 이 셀은 보고 기간을 계산합니다.
report_period = (
    f"{quality_df['Start_Date'].min().date()} ~ "
    f"{quality_df['End_Date'].max().date()}"
)

In [42]:
# Cell 42
# 이 셀은 KPI 요약에 필요한 값을 계산합니다.
kpi_summary = pd.Series(
    data={
        "보고기간": report_period,
        "평균수율": round(quality_df["수율검증값"].mean(), 2),
        "전체불량률": round(
            quality_df["총불량수"].sum() / quality_df["Input_Wafer_Qty"].sum() * 100,
            2,
        ),
        "검사Lot수": quality_df["Lot_ID"].nunique(),
        "주요불량유형": (
            quality_df["최다불량유형"]
            .replace(to_replace="없음", value=pd.NA)
            .dropna()
            .mode()
            .iloc[0]
        ),
        "수율미달Lot수": quality_df["수율미달여부"].sum(),
        "최저수율제품": quality_df.loc[
            quality_df["수율검증값"].idxmin(),
            "Product",
        ],
    }
)

In [43]:
# Cell 43
# 이 셀은 KPI 요약 결과를 확인합니다.
kpi_summary

보고기간        2026-03-01 ~ 2026-04-01
평균수율                          92.73
전체불량률                          3.62
검사Lot수                          150
주요불량유형                      Scratch
수율미달Lot수                        150
최저수율제품                       NAND_A
dtype: object

### 8단계. 제품별 품질 지표 산출

- 지시: `quality_df`를 `Product` 기준으로 묶어 제품별 검사 Lot 수, 평균 수율, 불량률을 계산하고 템플릿의 제품 순서인 DRAM_A, DRAM_B, Logic_A, NAND_A에 맞춰 `product_quality_summary`를 정리하세요.
- 이유: 템플릿의 제품별 품질 지표 표를 채우려면 제품 순서와 지표 계산 기준이 일치해야 하기 때문입니다.

In [44]:
# Cell 44
# 이 셀은 템플릿의 제품 순서를 리스트로 저장합니다.
product_order = ["DRAM_A", "DRAM_B", "Logic_A", "NAND_A"]

In [45]:
# Cell 45
# 이 셀은 Product 기준으로 제품별 검사 Lot 수, 평균 수율, 불량률을 계산합니다.
product_quality_summary = (
    quality_df
    .groupby(by="Product")
    .agg(
        검사Lot수=("Lot_ID", "nunique"),
        평균수율=("수율검증값", "mean"),
        총불량수=("총불량수", "sum"),
        총투입웨이퍼수=("Input_Wafer_Qty", "sum"),
    )
    .reindex(index=product_order)
)

In [46]:
# Cell 46
# 이 셀은 제품별 불량률을 계산하고 필요한 열만 정리합니다.
product_quality_summary["평균수율"] = product_quality_summary["평균수율"].round(2)
product_quality_summary["불량률"] = (
    product_quality_summary["총불량수"]
    / product_quality_summary["총투입웨이퍼수"]
    * 100
).round(2)

product_quality_summary = product_quality_summary[
    ["검사Lot수", "평균수율", "불량률"]
]

In [47]:
# Cell 47
# 이 셀은 제품별 품질 요약 결과를 확인합니다.
product_quality_summary

,검사Lot수,평균수율,불량률
Product,,,
DRAM_A,63,92.40,3.71
DRAM_B,32,94.09,2.39
Logic_A,22,91.91,4.78
NAND_A,33,92.57,3.90


### 9단계. 품질 이슈 해석용 보조 분석

- 지시: `quality_df`와 `defect_clean_df`를 사용하여 제품별·Fab별 평균 수율 차이, Final_Judgment 분포, 불량 유형별 총 불량 수와 심각도 분포를 요약하세요. 각 셀 단위로 파이썬 코드를 작성하세요.
- 이유: 템플릿의 분석 결과 요약 문장을 근거 있게 작성하려면 주요 품질 편차와 불량 특성을 확인해야 하기 때문입니다.

In [48]:
# Cell 48
# 이 셀은 제품별·Fab별 평균 수율을 계산합니다.
product_fab_yield_summary = (
    quality_df
    .groupby(by=["Product", "Fab"])
    .agg(평균수율=("수율검증값", "mean"))
    .round(2)
)

product_fab_yield_summary

평균수율
Product Fab        
DRAM_A  FAB1  91.32
        FAB2  93.45
DRAM_B  FAB1  93.29
        FAB2  95.13
Logic_A FAB1  93.00
        FAB2  91.29
NAND_A  FAB1  91.48
        FAB2  93.72

In [49]:
# Cell 49
# 이 셀은 제품별 평균 수율과 Fab별 평균 수율 차이를 보기 좋게 정리합니다.
product_fab_yield_pivot = product_fab_yield_summary.unstack(level="Fab")

product_fab_yield_pivot

평균수율       
Fab       FAB1   FAB2
Product              
DRAM_A   91.32  93.45
DRAM_B   93.29  95.13
Logic_A  93.00  91.29
NAND_A   91.48  93.72

In [50]:
# Cell 50
# 이 셀은 Final_Judgment 분포를 건수로 확인합니다.
final_judgment_count = quality_df["Final_Judgment"].value_counts().sort_index()

final_judgment_count

Final_Judgment
Hold      28
Pass      95
Review    27
Name: count, dtype: int64

In [51]:
# Cell 51
# 이 셀은 Final_Judgment 분포를 비율로 확인합니다.
final_judgment_ratio = (
    quality_df["Final_Judgment"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(other=100)
    .round(2)
)

final_judgment_ratio

Final_Judgment
Hold      18.67
Pass      63.33
Review    18.00
Name: proportion, dtype: float64

In [52]:
# Cell 52
# 이 셀은 불량 유형별 총 불량 수를 계산합니다.
defect_type_count = (
    defect_clean_df["Defect_Type"]
    .value_counts()
    .sort_values(ascending=False)
)

defect_type_count

Defect_Type
Scratch           28
Pattern bridge    21
Thickness fail    19
CD outlier        18
Overlay fail      17
Particle          17
Open defect       15
Name: count, dtype: int64

In [53]:
# Cell 53
# 이 셀은 불량 유형별 심각도 분포를 건수로 확인합니다.
defect_severity_count = pd.crosstab(
    index=defect_clean_df["Defect_Type"],
    columns=defect_clean_df["Severity"],
)

defect_severity_count

Severity,High,Low,Medium
Defect_Type,,,
CD outlier,3,9,6
Open defect,2,9,4
Overlay fail,4,8,5
Particle,3,10,4
Pattern bridge,1,12,8
Scratch,4,14,10
Thickness fail,2,10,7


In [54]:
# Cell 54
# 이 셀은 불량 유형별 심각도 분포를 비율로 확인합니다.
defect_severity_ratio = (
    pd.crosstab(
        index=defect_clean_df["Defect_Type"],
        columns=defect_clean_df["Severity"],
        normalize="index",
    )
    .mul(other=100)
    .round(2)
)

defect_severity_ratio

Severity,High,Low,Medium
Defect_Type,,,
CD outlier,16.67,50.00,33.33
Open defect,13.33,60.00,26.67
Overlay fail,23.53,47.06,29.41
Particle,17.65,58.82,23.53
Pattern bridge,4.76,57.14,38.10
Scratch,14.29,50.00,35.71
Thickness fail,10.53,52.63,36.84


### 10단계. 분석 결과 요약 문장 작성

- 지시: `kpi_summary`, `product_quality_summary`, 보조 분석 결과를 바탕으로 템플릿의 분석 결과 요약 칸에 넣을 3문장 이내의 품질 현황과 주의사항을 작성하세요.
- 이유: 숫자만 채운 보고서보다 주요 품질 리스크와 해석을 함께 제시해야 의사결정에 활용할 수 있기 때문입니다.

In [55]:
# Cell 55
# 이 셀은 템플릿의 분석 결과 요약 칸에 넣을 문장을 작성합니다.
analysis_summary = (
    f"보고 기간 {kpi_summary['보고기간']} 동안 총 {kpi_summary['검사Lot수']}개 Lot을 검사했으며, "
    f"평균 수율은 {kpi_summary['평균수율']}%, 전체 불량률은 {kpi_summary['전체불량률']}%입니다. "
    f"주요 불량 유형은 {kpi_summary['주요불량유형']}이며, 수율 미달 Lot은 "
    f"{kpi_summary['수율미달Lot수']}개로 확인되었습니다. "
    f"최저 수율 제품은 {kpi_summary['최저수율제품']}이므로 해당 제품과 관련 Fab의 공정 조건을 우선 점검해야 합니다."
)

analysis_summary

'보고 기간 2026-03-01 ~ 2026-04-01 동안 총 150개 Lot을 검사했으며, 평균 수율은 92.73%, 전체 불량률은 3.62%입니다. 주요 불량 유형은 Scratch이며, 수율 미달 Lot은 150개로 확인되었습니다. 최저 수율 제품은 NAND_A이므로 해당 제품과 관련 Fab의 공정 조건을 우선 점검해야 합니다.'

### 11단계. 보고서 템플릿 자동 채우기

- 지시: `report_template.xlsx`의 `품질_보고서` 시트에서 주요 KPI 요약 영역, 제품별 품질 지표 영역, 분석 결과 요약 영역을 계산 결과로 채우고 `반도체_품질_분석_보고서_완성본.xlsx`로 저장하세요.
- 이유: 계산한 지표를 템플릿에 반영한 완성본 파일을 만들어 제출 가능한 보고서 형태로 마무리해야 하기 때문입니다.

In [56]:
# Cell 56
# 이 셀은 품질_보고서 시트에 값을 입력할 대상 셀을 찾는 함수를 만듭니다.
def find_input_cell(ws, label):
    for row in ws.iter_rows():
        for cell in row:
            if cell.value == label:
                right_cell = ws.cell(row=cell.row, column=cell.column + 1)
                return right_cell.coordinate
    return None

In [57]:
# Cell 57 수정
# 이 셀은 작성일자와 주요 KPI 요약 영역을 올바른 값으로 다시 입력합니다.
from datetime import date

report_ws["B2"] = kpi_summary["보고기간"]
report_ws["D2"] = date.today()

report_ws["B5"] = kpi_summary["평균수율"] / 100
report_ws["D5"] = kpi_summary["전체불량률"] / 100

report_ws["B6"] = kpi_summary["검사Lot수"]
report_ws["D6"] = kpi_summary["주요불량유형"]

report_ws["B7"] = kpi_summary["수율미달Lot수"]
report_ws["D7"] = kpi_summary["최저수율제품"]

In [58]:
# Cell 58
# 이 셀은 제품별 품질 지표 영역을 템플릿의 고정 위치에 입력합니다.
product_row_map = {
    "DRAM_A": 11,
    "DRAM_B": 12,
    "Logic_A": 13,
    "NAND_A": 14,
}

for product, row in product_row_map.items():
    report_ws[f"B{row}"] = product_quality_summary.loc[product, "검사Lot수"]
    report_ws[f"C{row}"] = product_quality_summary.loc[product, "평균수율"]
    report_ws[f"D{row}"] = product_quality_summary.loc[product, "불량률"]

In [59]:
# Cell 59
# 이 셀은 분석 결과 요약 문장을 노란색 입력 영역에 입력합니다.
report_ws["A17"] = analysis_summary

In [60]:
# Cell 60
# 이 셀은 수정된 품질 분석 보고서를 다시 저장합니다.
template_wb.save(filename="반도체_품질_분석_보고서_완성본.xlsx")

### 12단계. 완성본 검증

- 지시: 저장된 `반도체_품질_분석_보고서_완성본.xlsx`를 다시 열어 주요 KPI 셀, 제품별 표, 분석 결과 요약 문장이 비어 있지 않은지 확인하세요.
- 이유: 파일 저장 과정에서 누락되거나 잘못 입력된 셀이 없는지 최종 확인해야 하기 때문입니다.

In [61]:
# Cell 61
# 이 셀은 저장된 완성본 파일을 다시 불러옵니다.
check_wb = load_workbook(filename="반도체_품질_분석_보고서_완성본.xlsx")
check_ws = check_wb["품질_보고서"]

In [62]:
# Cell 62
# 이 셀은 값이 비어 있는지 확인하는 함수를 만듭니다.
def is_filled(value):
    return value is not None and str(value).strip() != ""

In [63]:
# Cell 63
# 이 셀은 주요 KPI 셀이 비어 있지 않은지 고정 셀 주소 기준으로 확인합니다.
kpi_check = {
    "보고기간_B2": is_filled(check_ws["B2"].value),
    "평균수율_B5": is_filled(check_ws["B5"].value),
    "전체불량률_D5": is_filled(check_ws["D5"].value),
    "검사Lot수_B6": is_filled(check_ws["B6"].value),
    "주요불량유형_D6": is_filled(check_ws["D6"].value),
    "수율미달Lot수_B7": is_filled(check_ws["B7"].value),
    "최저수율제품_D7": is_filled(check_ws["D7"].value),
}

pd.Series(data=kpi_check)

보고기간_B2        True
평균수율_B5        True
전체불량률_D5       True
검사Lot수_B6      True
주요불량유형_D6      True
수율미달Lot수_B7    True
최저수율제품_D7      True
dtype: bool

In [64]:
# Cell 64
# 이 셀은 제품별 표의 값이 비어 있지 않은지 확인합니다.
product_table_check = {}

for product in product_order:
    for row in range(1, check_ws.max_row + 1):
        for col in range(1, check_ws.max_column + 1):
            if check_ws.cell(row=row, column=col).value == product:
                product_table_check[product] = all(
                    is_filled(check_ws.cell(row=row, column=col + i).value)
                    for i in range(1, 4)
                )

pd.Series(data=product_table_check)

DRAM_A     True
DRAM_B     True
Logic_A    True
NAND_A     True
dtype: bool

In [65]:
# Cell 65
# 이 셀은 분석 결과 요약 문장이 비어 있지 않은지 확인합니다.
summary_cell = find_input_cell(ws=check_ws, label="분석 결과 요약")

pd.Series(
    data={
        "분석결과요약_입력여부": is_filled(check_ws[summary_cell].value),
        "분석결과요약_내용": check_ws[summary_cell].value,
    }
)

분석결과요약_입력여부    False
분석결과요약_내용       None
dtype: object